# EgoRecover-JEPA - Baseline 1 at scale: frozen V-JEPA 2.1 on clean video
** | EPIC-KITCHENS-100 action anticipation | Kaggle or Colab | everything is backed up to Google Drive (rclone)**

**Before running**
1. Kaggle: *Internet ON* and a *GPU* (T4/P100); attach the secret `RCLONE_TOKEN` (Add-ons -> Secrets). Colab: GPU runtime and the secret `RCLONE_TOKEN` in the key panel (enable *Notebook access*).
2. Run the 8 code cells top to bottom. Cells 1-3 are quick setup, **cell 4 is the long one** (download + feature extraction), cells 5-8 are training, evaluation, figures and the report.

**How downloading was made fast**
- The official downloader uses a single connection with no resume (this is why one 6 GB video took ~3 h). Here, the newer *extension* videos come from a **HuggingFace mirror** (`awsaf49/epic_kitchens_100`, fast CDN, multi-connection), and several videos download **in parallel** while the GPU encodes the previous ones.
- Official validation videos only exist on the Bristol server; they use a **resumable multi-connection (HTTP Range) downloader**. A speed probe measures 1/4/16 connections and picks the best.
- Nothing needs to be kept: each video becomes a small **feature shard** (~70 MB) on Drive; the raw `.MP4` is also copied to Drive (needed later for the corruption experiments) and then deleted locally. 

**Resume:** the cells are safe to re-run. Finished shards on Drive are skipped, so after a disconnect just run cells 1, 2, 4 again (set `RUN_EXTRACTION = False` in cell 1 to only re-train/evaluate).

### 1) Setup: configuration, environment, logging, Google Drive

In [ ]:
# =====================================================================================================
# 1) SETUP  -  configuration, environment, logging, Google Drive (rclone)
#    Works on Kaggle (Internet ON + secret RCLONE_TOKEN attached) and on Colab (secret RCLONE_TOKEN in the key panel)
# =====================================================================================================
import os, sys, json, time, shutil, subprocess, pathlib, re, threading, glob, hashlib, traceback, math, warnings
warnings.filterwarnings("ignore")

# ------------------------------------------ CONFIG (edit here) ------------------------------------------
# data selection
N_EXT_VIDEOS        = 60      # extension videos from the FAST HuggingFace mirror; split into train / held-out BY VIDEO
HELDOUT_FRAC        = 0.20    # share of those videos held out for validation (video-level split)
OFFICIAL_VAL_VIDEOS = 8       # official EPIC-100 validation videos from the (slow) Bristol server; 0 = skip
EXTRA_DRIVE_VIDEOS  = {"P01_01": "1KbrRXj1lrCoF_ircAk8WHlrzpwpPoWcK"}   # videos already on your Drive (Drive file id) -> extra TRAIN video
MIN_MIN, MAX_MIN    = 3.0, 15.0                                       # only videos with this duration range (minutes)
# clips / encoder
OBS_SEC, TAU_A, NF, IMG, POOL = 4.0, 1.0, 16, 384, 4                  # 16 frames from 4 s ending 1 s before the action
MODEL = "apiantonio/vjepa2.1-vit-large-384"                           # frozen V-JEPA 2.1 ViT-L (HF port)
ENC_BS = 2                                                            # encoder batch size (GPU memory)
# probe training
EPOCHS, PROBE_BS, LR, WD = 30, 64, 1e-3, 0.05
SEEDS = [0, 1, 2]                                                     # main attentive probe is trained once per seed
# runtime / transfer
TIME_BUDGET_H       = 9.0     # stop starting new downloads after this many hours (everything is resumable)
HF_WORKERS, BRISTOL_WORKERS, UPLOAD_WORKERS = 3, 2, 3
MAX_VIDEOS_ON_DISK  = 6       # download backpressure (keeps local disk bounded)
BACKUP_RAW_VIDEOS   = True    # also copy the raw .MP4 files to Google Drive (needed again for the corruption experiments)
RUN_EXTRACTION      = True    # False = skip downloading/encoding and only train/evaluate on the shards already on Drive
# ---------------------------------------------------------------------------------------------------------

IN_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
TEMP = "/kaggle/temp" if IN_KAGGLE else "/content/temp"
RUN_ID    = "baseline1_" + time.strftime("%Y%m%d_%H%M%S")
RUN_DIR   = f"{WORK}/runs/{RUN_ID}"
EPIC_ROOT = f"{TEMP}/EPIC-KITCHENS"
for d in (RUN_DIR, EPIC_ROOT, f"{RUN_DIR}/figures", f"{RUN_DIR}/tables"): os.makedirs(d, exist_ok=True)

FEAT_TAG    = f"{MODEL.split('/')[-1]}_f{NF}_o{OBS_SEC}_p{POOL}_s{IMG}"
FEATS_LOCAL = f"{TEMP}/features/{FEAT_TAG}"; os.makedirs(FEATS_LOCAL, exist_ok=True)
DRIVE_ROOT   = "gdrive:EgoRecover-JEPA"
DRIVE_FEATS  = f"{DRIVE_ROOT}/features/{FEAT_TAG}"
DRIVE_VIDEOS = f"{DRIVE_ROOT}/EPIC-KITCHENS"
DRIVE_RUN    = f"{DRIVE_ROOT}/runs/{RUN_ID}"

# ---- logging: everything printed is also written to a log file that is backed up to Drive ----
class Tee:
    def __init__(self, stream, path): self.s, self.f = stream, open(path, "a", buffering=1)
    def write(self, x): self.s.write(x); self.f.write(x); return len(x)
    def flush(self): self.s.flush(); self.f.flush()
    def isatty(self): return False
if not isinstance(sys.stdout, Tee): sys.stdout = Tee(sys.stdout, f"{RUN_DIR}/console_log.txt")
print(f"run id: {RUN_ID} | platform: {'Kaggle' if IN_KAGGLE else 'Colab/other'} | temp: {TEMP}", flush=True)

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)

# ---- packages ----
print("installing packages ...", flush=True)
r = sh([sys.executable, "-m", "pip", "-q", "install", "-U", "transformers", "huggingface_hub", "hf_transfer", "requests", "pandas", "matplotlib", "tqdm"])
if r.returncode != 0: print("pip warning:", r.stderr[-400:])
try:
    import hf_transfer  # noqa: F401
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"          # multi-connection HuggingFace downloads
except ImportError:
    print("hf_transfer not available (downloads still work, just slower)")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HOME"] = f"{TEMP}/hf_cache"

FFMPEG = shutil.which("ffmpeg")
if FFMPEG is None:
    sh([sys.executable, "-m", "pip", "-q", "install", "imageio-ffmpeg"])
    import imageio_ffmpeg; FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

# ---- Google Drive through rclone (token from the RCLONE_TOKEN secret) ----
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception: pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception: pass
    return os.environ.get(name)

if shutil.which("rclone") is None:
    print("installing rclone ...", flush=True); sh("curl -s https://rclone.org/install.sh | bash")
assert shutil.which("rclone"), "rclone install failed - is Internet enabled for this notebook?"
_tok = get_secret("RCLONE_TOKEN")
assert _tok, "Secret RCLONE_TOKEN not found (Kaggle: Add-ons > Secrets; Colab: key icon). Enable it for this notebook."
_cfg = pathlib.Path.home() / ".config" / "rclone"; _cfg.mkdir(parents=True, exist_ok=True)
(_cfg / "rclone.conf").write_text(f"[gdrive]\ntype = drive\nscope = drive\ntoken = {json.dumps(json.loads(_tok))}\n")

def rc(args, timeout=None):
    return subprocess.run(["rclone"] + args, capture_output=True, text=True, timeout=timeout)

_r = rc(["lsd", "gdrive:"], timeout=120)
assert _r.returncode == 0, "rclone cannot reach Google Drive: " + _r.stderr[-300:]
for p in (DRIVE_FEATS, DRIVE_VIDEOS, DRIVE_RUN): rc(["mkdir", p])
print("Google Drive connected. Top-level folders:", [l.split()[-1] for l in _r.stdout.splitlines()][:12], flush=True)

def sync_run(note=""):
    """Back up everything produced in this run (logs, tables, figures, probes) to Google Drive."""
    r = rc(["copy", RUN_DIR, DRIVE_RUN, "--exclude", "*.part", "--transfers", "4"], timeout=1800)
    print(f"[backup] {note or 'run dir'} -> {DRIVE_RUN} | rc={r.returncode} {r.stderr[-150:]}", flush=True)

# ---- environment info ----
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENV = dict(python=sys.version.split()[0], torch=torch.__version__, device=DEVICE,
           gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
           disk_free_gb=round(shutil.disk_usage(TEMP).free / 1e9, 1), cpus=os.cpu_count())
try:
    import transformers; ENV["transformers"] = transformers.__version__
except Exception: pass
print("environment:", ENV, flush=True)
CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list, dict)) and k not in ("EXTRA_DRIVE_VIDEOS",)}
json.dump({"config": CONFIG, "env": ENV, "extra_drive_videos": list(EXTRA_DRIVE_VIDEOS)}, open(f"{RUN_DIR}/config.json", "w"), indent=2)
sync_run("setup")

In [ ]:
import sys
if sys.stdout.__class__.__name__ == "Tee": sys.stdout = sys.stdout.s     # restore the notebook's normal output
need = ["RUN_DIR", "rc", "sync_run", "TEMP", "PLAN", "ACTION_MAP", "N_VERB", "N_NOUN", "N_ACTION", "TR_CSV",
        "TP", "TPS", "F", "fetch_video", "process_video", "load_encoder", "upload"]
missing = [n for n in need if n not in globals()]
print("stdout:", type(sys.stdout).__name__)
print("missing variables:", missing or "none - cells 1-3 all ran")
if "PLAN" in globals(): print("plan:", len(PLAN), "videos | shards already done:", int(PLAN.shard_done.sum()))

In [ ]:
# ================= SAVE & VERIFY: back up the whole session to Google Drive, check it arrived, build a ZIP =================
# Prints a timestamped line at every step and writes SAVE_STATUS.txt to Drive, so you can confirm success even if the notebook output pane is blank.
import os, sys, json, time, glob, pickle, zipfile, subprocess
assert "RUN_DIR" in globals() and "rc" in globals(), "Variables from cell 1 are missing (session was restarted). Run cells 1-3 first."
FIX_MISSING_RAW = False   # True = also re-download + upload raw videos that are missing on Drive (can take long)
ZIP_PROBES      = True    # include trained probe weights in the zip (~70 MB each)
STATUS = []

def say(*a): print(time.strftime("%H:%M:%S"), *a, flush=True)
def step(name, ok, info="", level="FAIL"):
    tag = "OK" if ok else level                      # level "WARN" = worth knowing, does not mean the save failed
    STATUS.append((name, bool(ok) or level == "WARN", info, tag)); say(f"{tag:<4}", "|", name, "|", info)

def rc_stream(args, label, timeout=7200):
    """rclone with live progress lines (never silent)."""
    p = subprocess.Popen(["rclone"] + args + ["--stats", "20s", "--stats-one-line", "-v"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    t0, tail = time.time(), ""
    for line in p.stdout:
        line = line.strip(); tail = line or tail
        if "Transferred:" in line or "ERROR" in line: say(f"   [{label}] {line[-150:]}")
        if time.time() - t0 > timeout: p.kill(); break
    p.wait(); return p.returncode

def drive_files(path):
    r = rc(["lsjson", "-R", "--files-only", path], timeout=600)
    return {x["Path"]: x["Size"] for x in json.loads(r.stdout or "[]")} if r.returncode == 0 else {}

say("save cell started | run:", RUN_ID, "| run dir:", RUN_DIR)

# 1) in-memory state that exists only in this session
saved = []
os.makedirs(f"{RUN_DIR}/probes", exist_ok=True)
if "MODELS" in globals():
    import torch
    for sd, m in MODELS.items(): torch.save(m.state_dict(), f"{RUN_DIR}/probes/probe_attentive_seed{sd}.pt"); saved.append(f"probe_seed{sd}")
state = {n: globals()[n] for n in ["PLAN", "SUMMARY", "ALL_HIST", "HIST_DF", "LONG", "COMPARE", "CTX", "TOPK", "PERCLASS", "PERVIDEO", "CI", "FAILED", "DL_LOG"] if n in globals()}
pickle.dump(state, open(f"{RUN_DIR}/session_state.pkl", "wb")); saved += list(state)
open(f"{RUN_DIR}/RESUME_NOTES.txt", "w").write(
    "Continue in a new session: run cell 1, cell 2, cell 3; run cell 4 only if videos are still missing; then cells 5-8.\n"
    f"Features: {DRIVE_FEATS}\nRaw videos: {DRIVE_VIDEOS}\nThis run: {DRIVE_RUN}\nReload tables: pickle.load(open('session_state.pkl','rb'))\n")
step("session state saved", True, f"{len(saved)} objects: {saved[:6]}...")

# 2) feature shards
loc = sorted(os.path.basename(p) for p in glob.glob(f"{FEATS_LOCAL}/*.pt"))
if loc: rc_stream(["copy", FEATS_LOCAL, DRIVE_FEATS, "--transfers", "8"], "shards")
dr = drive_files(DRIVE_FEATS)
miss = [f for f in loc if f not in dr]
step("feature shards on Drive", not miss, f"{len(dr)} on Drive, {len(loc)} local, {len(miss)} local ones missing on Drive")

# 3) raw videos
left = glob.glob(f"{EPIC_ROOT}/**/*.MP4", recursive=True)
if left:
    say(f"uploading {len(left)} local raw video(s) that are not yet on Drive (large; progress below) ...")
    rc_stream(["copy", EPIC_ROOT, DRIVE_VIDEOS, "--include", "*.MP4", "--exclude", "*.part", "--transfers", "4", "--drive-chunk-size", "128M"], "videos")
have = {os.path.basename(p)[:-4] for p in drive_files(DRIVE_VIDEOS) if p.upper().endswith(".MP4")}
if "PLAN" in globals():
    mv = [v for v in PLAN.vid if v not in have]
    if mv and FIX_MISSING_RAW and "fetch_video" in globals():
        for _, row in PLAN[PLAN.vid.isin(mv)].iterrows():
            try:
                src = row["source"] if row["source"] in ("hf", "bristol", "drive_copyid") else "hf"
                p = fetch_video({**row.to_dict(), "source": src, "remote": row["remote"] if src == row["source"] else HF_FILES[row["vid"]]})
                upload(p, f"{DRIVE_VIDEOS}/{row['pid']}/videos", True); say("   re-uploaded", row["vid"])
            except Exception as e: say("   could not fix", row["vid"], repr(e)[:100])
        have = {os.path.basename(p)[:-4] for p in drive_files(DRIVE_VIDEOS) if p.upper().endswith(".MP4")}
        mv = [v for v in PLAN.vid if v not in have]
    step("raw videos on Drive", not mv, f"{len(have)} MP4 on Drive; {len(mv)} planned videos without a raw copy {mv[:8]}" + ("" if not mv else " (their feature shards exist, so training is not affected)"), level="WARN")
else:
    step("raw videos on Drive", True, f"{len(have)} MP4 on Drive")

# 4) the run folder: copy, then compare file names + sizes with Drive
rc_stream(["copy", RUN_DIR, DRIVE_RUN, "--exclude", "*.part", "--transfers", "4"], "run folder")
local = {os.path.relpath(os.path.join(r, f), RUN_DIR): os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(RUN_DIR) for f in fs}
dr = drive_files(DRIVE_RUN)
bad = [k for k, s in local.items() if k not in dr or (k != "console_log.txt" and dr[k] != s)]
step("run folder matches Drive", not bad, f"{len(local)} local files, {len(dr)} on Drive, {len(bad)} missing/different {bad[:5]}")

# 5) ZIP for your PC (+ copy on Drive in EgoRecover-JEPA/exports)
zip_path = f"{WORK}/{RUN_ID}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for r, _, fs in os.walk(RUN_DIR):
        for f in fs:
            if f.endswith(".pt") and not ZIP_PROBES: continue
            z.write(os.path.join(r, f), os.path.join(RUN_ID, os.path.relpath(os.path.join(r, f), RUN_DIR)))
rc_stream(["copy", zip_path, f"{DRIVE_ROOT}/exports"], "zip")
ex = drive_files(f"{DRIVE_ROOT}/exports").get(os.path.basename(zip_path))
step("zip on Drive", ex == os.path.getsize(zip_path), f"{os.path.basename(zip_path)}: {os.path.getsize(zip_path) / 1e6:.0f} MB local, {(ex or 0) / 1e6:.0f} MB on Drive")

# 6) verdict file (readable in the Drive web page even if the notebook shows nothing)
ok_all = all(ok for _, ok, _, _ in STATUS)
txt = [f"SAVE STATUS: {'ALL OK' if ok_all else 'SOME PROBLEMS'}", f"time: {time.strftime('%Y-%m-%d %H:%M:%S')}", f"run: {RUN_ID}", ""] + \
      [f"[{tag}] {n}: {i}" for n, ok, i, tag in STATUS] + \
      ["", "Where to look in Google Drive (account that owns the rclone token):", f"  EgoRecover-JEPA/runs/{RUN_ID}/   (results, figures, tables, probes, logs)",
       f"  EgoRecover-JEPA/exports/{os.path.basename(zip_path)}   (zip for your PC)", "  EgoRecover-JEPA/features/<model tag>/   (feature shards)", "  EgoRecover-JEPA/EPIC-KITCHENS/P*/videos/   (raw videos)"]
open(f"{RUN_DIR}/SAVE_STATUS.txt", "w").write("\n".join(txt))
for tgt in (DRIVE_RUN, DRIVE_ROOT):
    rc(["copyto", f"{RUN_DIR}/SAVE_STATUS.txt", f"{tgt}/SAVE_STATUS{'' if tgt == DRIVE_RUN else '_latest'}.txt"])
rc(["copy", f"{RUN_DIR}/console_log.txt", DRIVE_RUN])                       # final log incl. this cell
print("\n" + "\n".join(txt), flush=True)
print("\nlocal files:", sorted(os.listdir(RUN_DIR)), "| zip:", zip_path, flush=True)

In [ ]:
!ls -lah /kaggle/working/runs/*/ | head -40

In [ ]:
# ================= SAVE SESSION: back up everything to Google Drive + build a ZIP for your PC =================
# Run this in the SAME session after your cells (it needs the variables from cell 1 and, if present, cells 2-8).
import os, sys, json, time, shutil, zipfile, pickle, glob
assert "RUN_DIR" in globals() and "rc" in globals(), "Run cell 1 first (this cell uses its variables)."

# ---- options ----
FIX_MISSING_RAW = True    # re-fetch (fast HF mirror) + upload any raw .MP4 that is missing on Drive
ZIP_PROBES      = True    # include trained probe weights in the zip (~70 MB each)
ZIP_FEATURES    = False   # include the feature shards (~4.5 GB) in the zip; they are already on Drive
DOWNLOAD_ZIP    = True    # Colab only: start a browser download of the zip

sys.stdout.flush()
os.makedirs(f"{RUN_DIR}/probes", exist_ok=True)
CHECK = {}

# ---- 1) save in-memory state that exists only in this session ----
saved = []
if "MODELS" in globals():                                   # probes of ALL seeds (cell 5 only saved the first one)
    import torch
    for sd, m in MODELS.items():
        torch.save(m.state_dict(), f"{RUN_DIR}/probes/probe_attentive_seed{sd}.pt"); saved.append(f"probe seed {sd}")
names = ["PLAN", "SUMMARY", "ALL_HIST", "HIST_DF", "LONG", "COMPARE", "CTX", "TOPK", "PERCLASS", "PERVIDEO", "CI", "FAILED", "DL_LOG"]
state = {n: globals()[n] for n in names if n in globals()}
with open(f"{RUN_DIR}/session_state.pkl", "wb") as f: pickle.dump(state, f)
saved += list(state)
json.dump(dict(run_id=RUN_ID, saved_at=time.strftime("%Y-%m-%d %H:%M:%S"), saved_objects=saved, env=globals().get("ENV"), config=globals().get("CONFIG")),
          open(f"{RUN_DIR}/session_state.json", "w"), indent=2, default=str)
open(f"{RUN_DIR}/RESUME_NOTES.txt", "w").write(
    "How to continue in a new session:\n1) run cell 1 (setup)  2) run cell 2 (plan: reads finished shards/videos from Drive)\n"
    "3) run cell 3 (definitions)  4) run cell 4 only if videos are still missing  5) run cells 5-8 to retrain/evaluate.\n"
    f"Features: {DRIVE_FEATS}\nRaw videos: {DRIVE_VIDEOS}\nThis run: {DRIVE_RUN}\n"
    "To reload the saved tables: import pickle; state = pickle.load(open('session_state.pkl','rb'))\n")
print("saved session objects:", saved, flush=True)

# ---- 2) feature shards: make sure every local shard is on Drive, then verify ----
rc(["copy", FEATS_LOCAL, DRIVE_FEATS, "--transfers", "8"], timeout=3600)
chk = rc(["check", FEATS_LOCAL, DRIVE_FEATS, "--size-only", "--one-way"], timeout=1800)
n_local = len(glob.glob(f"{FEATS_LOCAL}/*.pt")); n_drive = len([x for x in rc(["lsf", DRIVE_FEATS, "--files-only"]).stdout.split() if x.endswith(".pt")])
CHECK["feature shards"] = (chk.returncode == 0, f"{n_local} local / {n_drive} on Drive")

# ---- 3) raw videos: upload leftovers, then re-fetch + upload any that are missing on Drive ----
def drive_raw():
    return {os.path.basename(p)[:-4] for p in rc(["lsf", DRIVE_VIDEOS, "-R", "--files-only"], timeout=600).stdout.split() if p.upper().endswith(".MP4")}
left = glob.glob(f"{EPIC_ROOT}/**/*.MP4", recursive=True)
if left:
    rc(["copy", EPIC_ROOT, DRIVE_VIDEOS, "--include", "*.MP4", "--exclude", "*.part", "--transfers", "4", "--drive-chunk-size", "128M"], timeout=7200)
have = drive_raw()
if "PLAN" in globals():
    missing = [v for v in PLAN.vid if v not in have]
    print(f"raw videos on Drive: {len(have)} | in plan: {len(PLAN)} | missing: {len(missing)} {missing[:10]}", flush=True)
    if missing and FIX_MISSING_RAW and "fetch_video" in globals():
        for _, row in PLAN[PLAN.vid.isin(missing)].iterrows():
            src = row["source"] if row["source"] in ("hf", "bristol", "drive_copyid") else ("hf" if row["vid"] in HF_FILES else None)
            if src is None: print(f"  cannot re-fetch {row['vid']} (no source)"); continue
            try:
                p = fetch_video({**row.to_dict(), "source": src, "remote": row["remote"] if row["source"] == src else HF_FILES[row["vid"]]})
                upload(p, f"{DRIVE_VIDEOS}/{row['pid']}/videos", True); print(f"  re-uploaded raw video {row['vid']}", flush=True)
            except Exception as e: print(f"  could not fix {row['vid']}: {repr(e)[:120]}")
        have = drive_raw()
    CHECK["raw videos"] = (all(v in have for v in PLAN.vid), f"{sum(v in have for v in PLAN.vid)} / {len(PLAN)} on Drive")

# ---- 4) the run folder (logs, tables, figures, report, probes, state) ----
sync_run("session save")
listing = rc(["lsf", DRIVE_RUN, "-R", "--files-only"], timeout=300).stdout.split()
CHECK["run folder"] = (all(k in listing for k in ("report.md", "console_log.txt", "session_state.pkl")) if os.path.exists(f"{RUN_DIR}/report.md") else
                       "console_log.txt" in listing, f"{len(listing)} files on Drive")

# ---- 5) ZIP for your PC (also copied to Drive: EgoRecover-JEPA/exports) ----
zip_path = f"{WORK}/{RUN_ID}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for root, _, files in os.walk(RUN_DIR):
        for fn in files:
            p = os.path.join(root, fn)
            if fn.endswith(".pt") and not ZIP_PROBES: continue
            z.write(p, os.path.relpath(p, os.path.dirname(RUN_DIR)))
    if ZIP_FEATURES:
        for p in glob.glob(f"{FEATS_LOCAL}/*.pt"): z.write(p, f"features/{os.path.basename(p)}")
rc(["copy", zip_path, f"{DRIVE_ROOT}/exports"], timeout=3600)
CHECK["zip"] = (os.path.exists(zip_path), f"{os.path.getsize(zip_path) / 1e6:.0f} MB -> {zip_path}")

print("\n==================== SAVE CHECK ====================")
for k, (ok, info) in CHECK.items(): print(f"  [{'OK' if ok else 'PROBLEM'}] {k}: {info}")
print(f"\nDrive: EgoRecover-JEPA/runs/{RUN_ID}  |  zip copy: EgoRecover-JEPA/exports/{os.path.basename(zip_path)}")
if IN_KAGGLE:
    print("Kaggle: the zip is in /kaggle/working. To download it to your PC use the notebook's Output panel (right sidebar), "
          "and press 'Save Version' if you want Kaggle to keep /kaggle/working after the session ends. The Drive copy is already safe.")
elif DOWNLOAD_ZIP:
    try:
        from google.colab import files; files.download(zip_path)
    except Exception as e: print("Colab download not started:", e)

### 2) Data plan: which videos, from where, what is already done

In [ ]:
# =====================================================================================================
# 2) DATA PLAN  -  metadata, which videos exist where, selection, Drive inventory (resume support)
# =====================================================================================================
import numpy as np, pandas as pd

ANN_DIR, DLM_DIR = f"{TEMP}/annotations", f"{TEMP}/dl_meta"
for url, d in (("https://github.com/epic-kitchens/epic-kitchens-100-annotations", ANN_DIR),
               ("https://github.com/epic-kitchens/epic-kitchens-download-scripts", DLM_DIR)):
    if not os.path.exists(d):
        r = sh(["git", "clone", "-q", "--depth", "1", url, d]); assert r.returncode == 0, r.stderr

def to_sec(ts):
    h, m, s = ts.split(":"); return int(h) * 3600 + int(m) * 60 + float(s)

TR_CSV = pd.read_csv(f"{ANN_DIR}/EPIC_100_train.csv"); VA_CSV = pd.read_csv(f"{ANN_DIR}/EPIC_100_validation.csv")
for _df in (TR_CSV, VA_CSV): _df["t"] = _df["start_timestamp"].map(to_sec)
VINFO = pd.read_csv(f"{ANN_DIR}/EPIC_100_video_info.csv").set_index("video_id")
SPLITS = pd.read_csv(f"{DLM_DIR}/data/epic_100_splits.csv")
S55 = pd.read_csv(f"{DLM_DIR}/data/epic_55_splits.csv").set_index("video_id")["split"].to_dict()
_md5 = pd.read_csv(f"{DLM_DIR}/data/md5.csv"); MD5 = {(a, str(b)): c for a, b, c in zip(_md5.file_remote_path, _md5.version, _md5.md5)}

# label spaces (from the FULL official train csv, so heads are the same size as in the paper protocol)
ACTION_MAP = {p: i for i, p in enumerate(sorted(set(zip(TR_CSV.verb_class.astype(int), TR_CSV.noun_class.astype(int)))))}
N_VERB, N_NOUN, N_ACTION = int(TR_CSV.verb_class.max()) + 1, int(TR_CSV.noun_class.max()) + 1, len(ACTION_MAP)
print(f"label space: {N_VERB} verbs | {N_NOUN} nouns | {N_ACTION} verb-noun actions", flush=True)

MIN_T = TAU_A + OBS_SEC                                   # a full 4 s observation window must exist before the action
_use = lambda df: df[df.t >= MIN_T]
CLIPS = {**_use(TR_CSV).groupby("video_id").size().to_dict(), **_use(VA_CSV).groupby("video_id").size().to_dict()}
OFFICIAL_SPLIT = {**{v: "train" for v in TR_CSV.video_id.unique()}, **{v: "val" for v in VA_CSV.video_id.unique()}}

V = pd.DataFrame({"vid": list(OFFICIAL_SPLIT)})
V["pid"] = V.vid.str.split("_").str[0]; V["ext"] = V.vid.str.split("_").str[1].str.len() == 3
V["official"] = V.vid.map(OFFICIAL_SPLIT); V["clips"] = V.vid.map(CLIPS).fillna(0).astype(int)
V["minutes"] = V.vid.map(VINFO["duration"]) / 60

# ---- where can each video be fetched from? ----
HF_REPO = "awsaf49/epic_kitchens_100"                     # community mirror of the extension videos (fast CDN)
HF_FILES = {}
try:
    from huggingface_hub import HfApi
    HF_FILES = {os.path.splitext(os.path.basename(f))[0]: f for f in HfApi().list_repo_files(HF_REPO, repo_type="dataset") if f.lower().endswith(".mp4")}
except Exception as e:
    print("[warn] could not list the HuggingFace mirror:", repr(e)[:200])
print(f"HuggingFace mirror: {len(HF_FILES)} videos available", flush=True)

BASE55 = "https://data.bris.ac.uk/datasets/3h91syskeag572hl6tvuovwv4d"; BASE100 = "https://data.bris.ac.uk/datasets/2g1n6qdydwa9u22shpxqzp0t8m"
def bristol_url(vid, pid, ext):
    if ext: return f"{BASE100}/{pid}/videos/{vid}.MP4", MD5.get((f"{pid}/videos/{vid}.MP4", "100"))
    return f"{BASE55}/videos/{S55[vid]}/{pid}/{vid}.MP4", MD5.get((f"videos/{S55[vid]}/{pid}/{vid}.MP4", "55"))

def round_robin(df, n):
    """Pick n videos spread across participants (shortest videos of each participant first)."""
    groups = [g.sort_values("minutes")["vid"].tolist() for _, g in df.groupby("pid")]
    out, i = [], 0
    while len(out) < n and any(groups):
        g = groups[i % len(groups)]
        if g: out.append(g.pop(0))
        i += 1
    return out

in_range = V.minutes.between(MIN_MIN, MAX_MIN)
cand_ext = V[V.ext & (V.official == "train") & in_range & (V.clips >= 20)]
if HF_FILES: cand_ext = cand_ext[cand_ext.vid.isin(HF_FILES)]
chosen = round_robin(cand_ext, N_EXT_VIDEOS)
_k = max(int(round(1 / HELDOUT_FRAC)), 2)
heldout = set(chosen[::_k])
official_val = round_robin(V[(V.official == "val") & in_range & (V.clips >= 15)], OFFICIAL_VAL_VIDEOS) if OFFICIAL_VAL_VIDEOS else []

# ---- Drive inventory: finished feature shards (name = <split>__<video>.pt) and backed-up raw videos ----
_ls = rc(["lsf", DRIVE_FEATS, "--files-only"], timeout=300).stdout.split()
SHARD_SPLIT = {n.split("__")[1][:-3]: n.split("__")[0] for n in _ls if n.endswith(".pt") and "__" in n}
DRIVE_RAW = set(os.path.basename(p)[:-4] for p in rc(["lsf", DRIVE_VIDEOS, "-R", "--files-only"], timeout=600).stdout.split() if p.upper().endswith(".MP4"))
print(f"Drive inventory: {len(SHARD_SPLIT)} finished feature shards | {len(DRIVE_RAW)} raw videos backed up", flush=True)

rows = []
def add(vid, split, source, remote=None, md5=None):
    r = V[V.vid == vid].iloc[0]
    est = r.minutes * 60 * (1.3 if r.ext else 3.0) / 1000                        # rough GB estimate
    rows.append(dict(vid=vid, pid=r.pid, split=split, source=source, remote=remote, md5=md5, minutes=round(r.minutes, 1),
                     clips=int(r.clips), est_gb=round(est, 2), csv="train" if r.official == "train" else "val"))
for vid in chosen:
    pid = vid.split("_")[0]
    if HF_FILES: add(vid, "val_heldout" if vid in heldout else "train", "hf", HF_FILES[vid])
    else:
        u, m = bristol_url(vid, pid, True); add(vid, "val_heldout" if vid in heldout else "train", "bristol", u, m)
for vid in official_val:
    pid = vid.split("_")[0]; u, m = bristol_url(vid, pid, False); add(vid, "val_official", "bristol", u, m)
for vid, fid in EXTRA_DRIVE_VIDEOS.items():
    if vid in V.vid.values and vid not in chosen: add(vid, "train", "drive_copyid", fid)

PLAN = pd.DataFrame(rows).drop_duplicates("vid").reset_index(drop=True)
for i, r in PLAN.iterrows():                                                   # reuse what is already on Drive
    if r.vid in SHARD_SPLIT: PLAN.loc[i, "split"] = SHARD_SPLIT[r.vid]         # keep the split a finished shard was made with
    if r.vid in DRIVE_RAW and r.source in ("hf", "bristol"): PLAN.loc[i, "source"] = "drive_raw"
PLAN["shard_done"] = PLAN.vid.isin(SHARD_SPLIT)
PLAN.to_csv(f"{RUN_DIR}/tables/plan.csv", index=False)

S = PLAN.groupby("split").agg(videos=("vid", "size"), clips=("clips", "sum"), est_gb=("est_gb", "sum")).round(1)
print("\nPLAN (videos / usable clips / estimated GB to download):\n", S.to_string())
print("\nby source:", PLAN.groupby("source").size().to_dict(), "| already finished:", int(PLAN.shard_done.sum()), "/", len(PLAN))
print("\nfirst rows:\n", PLAN.head(8)[["vid", "split", "source", "minutes", "clips", "est_gb", "shard_done"]].to_string(index=False), flush=True)
assert len(PLAN) > 0, "Empty plan - check the filters in the config"
sync_run("data plan")

### 3) Library: fast downloader, one-pass frame decoding, frozen encoder

In [ ]:
# =====================================================================================================
# 3) LIBRARY  -  fast resumable downloader, one-pass frame decoding, frozen V-JEPA 2.1 encoder, shard writer
#    (definitions only; nothing heavy runs here)
# =====================================================================================================
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

STOP = threading.Event()          # set to abort all downloads (time budget / interrupt)
PROG = {}                         # dest -> [bytes_done, total]  (progress of multi-connection downloads)
_tls = threading.local()

def _sess():
    s = getattr(_tls, "s", None)
    if s is None:
        s = _tls.s = requests.Session(); s.headers["User-Agent"] = "Mozilla/5.0 (EgoRecover-JEPA downloader)"
    return s

def remote_info(url, timeout=60):
    """(total_bytes, server_supports_range)"""
    r = _sess().get(url, headers={"Range": "bytes=0-0"}, stream=True, timeout=timeout)
    try:
        if r.status_code == 206: return int(r.headers["Content-Range"].split("/")[-1]), True
        if r.status_code == 200: return int(r.headers.get("Content-Length", 0)), False
        raise IOError(f"HTTP {r.status_code} for {url}")
    finally:
        r.close()

def _fetch_chunk(url, fd, start, end, timeout, retries):
    last = None
    for attempt in range(retries):
        try:
            if STOP.is_set(): raise InterruptedError("stopped")
            r = _sess().get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=timeout)
            if r.status_code != 206: raise IOError(f"HTTP {r.status_code}")
            pos = start
            for buf in r.iter_content(1 << 20):
                if STOP.is_set(): raise InterruptedError("stopped")
                os.pwrite(fd, buf, pos); pos += len(buf)
            r.close()
            if pos != end + 1: raise IOError(f"short chunk ({pos - start}/{end + 1 - start} bytes)")
            return end - start + 1
        except InterruptedError: raise
        except Exception as e:
            last = e; time.sleep(min(2 ** attempt, 20))
    raise IOError(f"chunk {start}-{end} failed after {retries} tries: {last}")

def fast_http_download(url, dest, conns=16, chunk_mb=16, timeout=60, retries=6):
    """Resumable multi-connection download (HTTP Range). Falls back to one stream if the server has no Range support."""
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    total, rng = remote_info(url)
    if os.path.exists(dest) and os.path.getsize(dest) == total:
        return dict(bytes=total, secs=0.0, conns=0, skipped=True)
    part, state, chunk = dest + ".part", dest + ".state", chunk_mb << 20
    t0 = time.time(); PROG[dest] = [0, total]
    try:
        if not rng or conns <= 1:                                   # single stream
            with _sess().get(url, stream=True, timeout=timeout) as r, open(part, "wb") as f:
                r.raise_for_status()
                for buf in r.iter_content(4 << 20):
                    if STOP.is_set(): raise InterruptedError("stopped")
                    f.write(buf); PROG[dest][0] += len(buf)
            n_conn = 1
        else:
            n = (total + chunk - 1) // chunk; done = set()
            if os.path.exists(state) and os.path.exists(part):
                try:
                    st = json.load(open(state))
                    if st["total"] == total and st["chunk"] == chunk: done = set(st["done"])
                except Exception: done = set()
            if not done or not os.path.exists(part):
                done = set()
                with open(part, "wb") as f: f.truncate(total)
            PROG[dest][0] = sum(min(chunk, total - i * chunk) for i in done)
            fd = os.open(part, os.O_WRONLY)
            try:
                with ThreadPoolExecutor(conns) as ex:
                    futs = {ex.submit(_fetch_chunk, url, fd, i * chunk, min((i + 1) * chunk, total) - 1, timeout, retries): i
                            for i in range(n) if i not in done}
                    try:
                        for k, fu in enumerate(as_completed(futs), 1):
                            i = futs[fu]; fu.result(); done.add(i); PROG[dest][0] += min(chunk, total - i * chunk)
                            if k % 8 == 0: json.dump(dict(total=total, chunk=chunk, done=sorted(done)), open(state, "w"))
                    except BaseException:
                        ex.shutdown(wait=False, cancel_futures=True); raise
            finally:
                os.close(fd)
                if STOP.is_set() or len(done) < n: json.dump(dict(total=total, chunk=chunk, done=sorted(done)), open(state, "w"))
            n_conn = conns
        if os.path.getsize(part) != total: raise IOError("size mismatch after download")
        os.replace(part, dest)
        if os.path.exists(state): os.remove(state)
        return dict(bytes=total, secs=time.time() - t0, conns=n_conn, skipped=False)
    finally:
        PROG.pop(dest, None)

def md5_of(path, block=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(block), b""): h.update(b)
    return h.hexdigest()

def _probe_one(url, start, size, timeout):
    r = _sess().get(url, headers={"Range": f"bytes={start}-{start + size - 1}"}, stream=True, timeout=timeout)
    n = sum(len(b) for b in r.iter_content(1 << 20)); r.close(); return n

def probe_speed(url, conns_list=(1, 4, 16), mb_each=4, timeout=60):
    """Measure aggregate MB/s with 1 / 4 / 16 parallel connections and return ({conns: MB/s}, best_conns)."""
    total, rng = remote_info(url)
    if not rng: return {1: None}, 1
    res = {}
    for c in conns_list:
        offs = [((i * 97 + 13) << 20) % max(total - (mb_each << 20), 1) for i in range(c)]
        t0 = time.time()
        with ThreadPoolExecutor(c) as ex: got = sum(ex.map(lambda o: _probe_one(url, o, mb_each << 20, timeout), offs))
        res[c] = got / max(time.time() - t0, 1e-6) / 1e6
    best = max(res.values()); pick = min(c for c, s in res.items() if s >= 0.9 * best)
    return res, pick

def vlocal(pid, vid): return f"{EPIC_ROOT}/{pid}/videos/{vid}.MP4"

def fetch_video(row, bristol_conns=16):
    """Bring one video to local disk from the best source; returns the local path."""
    pid, vid, src = row["pid"], row["vid"], row["source"]
    dest = vlocal(pid, vid); os.makedirs(os.path.dirname(dest), exist_ok=True)
    if src == "hf":
        from huggingface_hub import hf_hub_download
        err = None
        for a in range(4):
            try: return hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=row["remote"], local_dir=EPIC_ROOT)
            except Exception as e: err = e; time.sleep(5 * (a + 1))
        raise err
    if src == "drive_raw":
        r = rc(["copy", f"{DRIVE_VIDEOS}/{pid}/videos/{vid}.MP4", os.path.dirname(dest), "--transfers", "4"], timeout=7200)
        if r.returncode != 0 or not os.path.exists(dest): raise IOError("rclone copy failed: " + r.stderr[-200:])
        return dest
    if src == "drive_copyid":
        r = rc(["backend", "copyid", "gdrive:", row["remote"], dest], timeout=7200)
        if r.returncode != 0 or not os.path.exists(dest): raise IOError("rclone copyid failed: " + r.stderr[-200:])
        return dest
    if src == "bristol":
        fast_http_download(row["remote"], dest, conns=bristol_conns)
        if row.get("md5") and isinstance(row["md5"], str):
            if md5_of(dest) != row["md5"]:
                os.remove(dest); raise IOError(f"md5 mismatch for {vid}")
        return dest
    raise ValueError(f"unknown source {src}")

# ----------------------------------------- frames: ONE decoding pass per video -----------------------------------------
FPS_X = NF / OBS_SEC                      # 4 frames per second -> a 16-frame clip spans 4 s
FRAME_BYTES = IMG * IMG * 3

def decode_video(video, out_path):
    """Decode the whole video once to a flat uint8 file [n, IMG, IMG, 3] at FPS_X (short side -> IMG, centre crop)."""
    cmd = [FFMPEG, "-hide_banner", "-loglevel", "error", "-nostdin", "-i", video, "-an", "-sn", "-threads", "0",
           "-vf", f"fps={FPS_X},scale=-2:{IMG}:flags=bilinear,crop={IMG}:{IMG}",
           "-f", "rawvideo", "-pix_fmt", "rgb24", "-y", out_path]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError("ffmpeg failed: " + r.stderr[-400:])
    return os.path.getsize(out_path) // FRAME_BYTES

def clip_index(t_start, n_frames):
    """Frame indices of the NF-frame observation window that ENDS tau_a seconds before the action starts."""
    last = int(round((t_start - TAU_A) * FPS_X)); first = last - (NF - 1)
    idx = np.clip(np.arange(first, last + 1), 0, n_frames - 1)
    return idx

# ----------------------------------------- TORCH PART (encoder) -----------------------------------------
import torch, torch.nn.functional as F
ENC, _LAYOUT = None, ["BTCHW"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1); _STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)
TP, TPS = NF // 2, (IMG // 16 // POOL) ** 2           # temporal steps (tubelet=2) and pooled spatial tokens per step
N_TOK = TP * TPS                                      # 288 tokens per clip with the defaults

def load_encoder():
    global ENC
    if ENC is not None: return ENC
    from transformers import AutoModel
    print("loading frozen V-JEPA 2.1 ...", flush=True)
    try:    m = AutoModel.from_pretrained(MODEL, trust_remote_code=True, dtype=torch.float16)
    except TypeError: m = AutoModel.from_pretrained(MODEL, trust_remote_code=True, torch_dtype=torch.float16)
    ENC = m.to(DEVICE).eval()
    for p in ENC.parameters(): p.requires_grad = False
    return ENC

@torch.no_grad()
def encode_batch(frames_u8):
    """np.uint8 [B,NF,IMG,IMG,3] -> pooled tokens fp16 [B, N_TOK, D] (time-major: step*TPS + spatial)."""
    x = torch.from_numpy(frames_u8).to(DEVICE).permute(0, 1, 4, 2, 3).float() / 255.0
    x = ((x - _MEAN.to(DEVICE)) / _STD.to(DEVICE)).half()                       # [B,T,C,H,W]
    B, T, C, H, W = x.shape
    expected = (T // 2) * (H // 16) * (W // 16); err = None
    for lay in [_LAYOUT[0]] + [l for l in ("BTCHW", "BCTHW", "BTHWC") if l != _LAYOUT[0]]:
        try:
            xin = x if lay == "BTCHW" else x.permute(0, 2, 1, 3, 4) if lay == "BCTHW" else x.permute(0, 1, 3, 4, 2)
            try:    out = ENC(pixel_values_videos=xin)
            except TypeError: out = ENC(xin)
            tok = out.last_hidden_state if hasattr(out, "last_hidden_state") else (out[0] if isinstance(out, (tuple, list)) else out)
            if tok.dim() == 3 and tok.shape[1] == expected: _LAYOUT[0] = lay; break
            err = f"{lay}: got {tuple(tok.shape)}, expected N={expected}"
        except Exception as e: err = f"{lay}: {type(e).__name__}: {e}"
    else: raise RuntimeError(f"encoder forward failed -> {err}")
    D = tok.shape[-1]
    t = tok.reshape(B * (T // 2), H // 16, W // 16, D).permute(0, 3, 1, 2).float()
    return F.avg_pool2d(t, POOL).permute(0, 2, 3, 1).reshape(B, -1, D).half()

def clip_rows(vid, csv_name):
    df = TR_CSV if csv_name == "train" else VA_CSV
    return df[(df.video_id == vid) & (df.t >= MIN_T)].sort_values("t").reset_index(drop=True)

def process_video(vid, path, split, csv_name):
    """decode once -> slice clip windows -> encode with the frozen encoder -> save one feature shard. Returns (shard_path, n_clips)."""
    rows = clip_rows(vid, csv_name)
    if len(rows) == 0: raise RuntimeError(f"{vid}: no usable clips")
    tmp = f"{TEMP}/frames_{vid}.u8"
    try:
        n = decode_video(path, tmp)
        expected = VINFO.loc[vid, "duration"] * FPS_X
        if n < 0.97 * expected: raise RuntimeError(f"{vid}: video looks truncated ({n} of ~{int(expected)} frames)")
        mm = np.memmap(tmp, dtype=np.uint8, mode="r", shape=(n, IMG, IMG, 3))
        xs = []
        for b in range(0, len(rows), ENC_BS):
            batch = np.stack([np.asarray(mm[clip_index(t, n)]) for t in rows.t.values[b:b + ENC_BS]])
            xs.append(encode_batch(batch).cpu())
        del mm
    finally:
        if os.path.exists(tmp): os.remove(tmp)
    shard = dict(x=torch.cat(xs), video_id=vid, split=split,
                 meta=dict(narration_id=rows.narration_id.tolist(), verb_class=rows.verb_class.astype(int).tolist(),
                           noun_class=rows.noun_class.astype(int).tolist(), verb=rows.verb.tolist(), noun=rows.noun.tolist(),
                           t=rows.t.astype(float).tolist()))
    out = f"{FEATS_LOCAL}/{split}__{vid}.pt"; torch.save(shard, out)
    return out, len(rows)

def upload(src, dst_dir, delete_after=False, release=None):
    try:
        r = rc(["copy", src, dst_dir, "--drive-chunk-size", "128M", "--transfers", "4"], timeout=7200)
        ok = r.returncode == 0
        if not ok: print(f"[upload failed] {os.path.basename(src)}: {r.stderr[-200:]}", flush=True)
        return ok
    finally:
        if delete_after and os.path.exists(src): os.remove(src)
        if release is not None: release()
print("library ready | tokens per clip:", N_TOK, f"({TP} time steps x {TPS} spatial)", flush=True)

### 4) Run: download -> features -> Drive (resumable, parallel)

In [ ]:
# =====================================================================================================
# 4) RUN  -  download -> frozen-encoder features -> Google Drive   (resumable; re-run this cell any time)
#    Videos are downloaded in parallel (fast HF mirror + multi-connection Bristol downloader) while the GPU encodes
#    the previous ones. Each finished video becomes one small feature shard on Drive; raw videos are backed up too.
# =====================================================================================================
STOP.clear()
_local_done = {f.split("__")[1][:-3] for f in os.listdir(FEATS_LOCAL) if f.endswith(".pt") and "__" in f}     # shards made earlier in this session
PLAN["shard_done"] = PLAN["shard_done"] | PLAN.vid.isin(_local_done)
todo = PLAN[~PLAN.shard_done].reset_index(drop=True)
DL_LOG, FAILED = [], []

if not RUN_EXTRACTION:
    print("RUN_EXTRACTION=False -> skipping download/encoding (using shards already on Drive).")
elif todo.empty:
    print("All feature shards of the plan are already on Drive - nothing to extract.")
else:
    assert DEVICE == "cuda", "A GPU is required for feature extraction (Runtime/Accelerator settings)."
    # ---- pick the number of parallel connections for the slow Bristol server ----
    bristol_conns = 16
    _b = todo[todo.source == "bristol"]
    if len(_b):
        try:
            _res, bristol_conns = probe_speed(_b.iloc[0]["remote"])
            print("Bristol speed probe, MB/s with 1 / 4 / 16 connections:", {k: (round(v, 2) if v else None) for k, v in _res.items()},
                  "-> using", bristol_conns, "connections per video", flush=True)
        except Exception as e:
            bristol_conns = 8; print("[warn] speed probe failed, using 8 connections:", repr(e)[:150])

    ex_hf, ex_bs, ex_up = ThreadPoolExecutor(HF_WORKERS), ThreadPoolExecutor(BRISTOL_WORKERS), ThreadPoolExecutor(UPLOAD_WORKERS)
    slots = threading.Semaphore(MAX_VIDEOS_ON_DISK)
    deadline = time.time() + TIME_BUDGET_H * 3600
    t_run = time.time(); REPORT_STOP = threading.Event()

    def dl_task(row):
        slots.acquire()
        try:
            while shutil.disk_usage(TEMP).free < 12e9 and not STOP.is_set(): time.sleep(20)     # keep room for decoding
            if STOP.is_set() or time.time() > deadline:
                slots.release(); return row, None, "skipped (time budget / stopped)"
            t0 = time.time(); path = fetch_video(row, bristol_conns)
            return row, path, dict(secs=time.time() - t0, gb=os.path.getsize(path) / 1e9)
        except BaseException as e:
            slots.release(); return row, None, repr(e)[:300]

    def reporter():
        while not REPORT_STOP.wait(60):
            act = [f"{os.path.basename(k)} {100 * v[0] / max(v[1], 1):.0f}%" for k, v in list(PROG.items())]
            print(f"  [status {int((time.time() - t_run) / 60)} min] finished {len(DL_LOG)}/{len(todo)} | failed {len(FAILED)} | "
                  f"bristol downloads: {act or '-'} | disk free {shutil.disk_usage(TEMP).free / 1e9:.0f} GB", flush=True)
    threading.Thread(target=reporter, daemon=True).start()

    order = pd.concat([todo[todo.source == "bristol"], todo[todo.source != "bristol"]])      # slow server first, overlapped with fast ones
    futs = [(ex_bs if r["source"] == "bristol" else ex_hf).submit(dl_task, r.to_dict()) for _, r in order.iterrows()]
    print(f"submitted {len(futs)} videos | HF workers {HF_WORKERS} | Bristol workers {BRISTOL_WORKERS} | time budget {TIME_BUDGET_H} h", flush=True)

    load_encoder()
    try:
        for fut in as_completed(futs):
            row, path, info = fut.result(); vid = row["vid"]
            if path is None:
                FAILED.append((vid, info)); print(f"[not done] {vid}: {info}", flush=True); continue
            t0 = time.time()
            try:
                shard, n_clips = process_video(vid, path, row["split"], row["csv"])
                ex_up.submit(upload, shard, DRIVE_FEATS)
                DL_LOG.append(dict(vid=vid, split=row["split"], source=row["source"], gb=round(info["gb"], 3), dl_secs=round(info["secs"], 1),
                                   dl_MBps=round(info["gb"] * 1000 / max(info["secs"], 1e-3), 1), clips=n_clips, encode_secs=round(time.time() - t0, 1)))
                print(f"[{len(DL_LOG)}/{len(todo)}] {vid:<8} {row['split']:<12} {n_clips:>4} clips | {info['gb']:.2f} GB in {info['secs']:.0f}s "
                      f"({DL_LOG[-1]['dl_MBps']} MB/s) | encoded in {time.time() - t0:.0f}s", flush=True)
                if BACKUP_RAW_VIDEOS and row["source"] in ("hf", "bristol", "drive_copyid"):
                    ex_up.submit(upload, path, f"{DRIVE_VIDEOS}/{row['pid']}/videos", True, slots.release)   # frees the disk slot afterwards
                else:
                    os.remove(path); slots.release()
            except Exception as e:
                traceback.print_exc(); FAILED.append((vid, repr(e)[:300]))
                if os.path.exists(path): os.remove(path)
                slots.release()
    except KeyboardInterrupt:
        print("interrupted - stopping downloads; finishing uploads ...", flush=True); STOP.set()
    finally:
        REPORT_STOP.set()
        for e in (ex_hf, ex_bs): e.shutdown(wait=False, cancel_futures=True)
        ex_up.shutdown(wait=True)
        globals()["ENC"] = None; torch.cuda.empty_cache()

    LOG = pd.DataFrame(DL_LOG); LOG.to_csv(f"{RUN_DIR}/tables/download_log.csv", index=False)
    print(f"\nextraction finished in {(time.time() - t_run) / 60:.1f} min | videos done {len(LOG)} | failed/skipped {len(FAILED)}")
    if len(LOG):
        print(f"downloaded {LOG.gb.sum():.1f} GB | clips encoded {LOG.clips.sum()} | mean download speed per video {LOG.dl_MBps.mean():.1f} MB/s")
        print(LOG.groupby("source").agg(videos=("vid", "size"), gb=("gb", "sum"), MBps=("dl_MBps", "mean")).round(1).to_string())
    if FAILED:
        pd.DataFrame(FAILED, columns=["vid", "reason"]).to_csv(f"{RUN_DIR}/tables/failed_videos.csv", index=False)
        print("not completed (just re-run THIS cell to retry):", [v for v, _ in FAILED][:20])
sync_run("extraction")

### 5) Baseline 1: train and evaluate the probe (3 seeds, every epoch logged)

In [ ]:
# =====================================================================================================
# 5) BASELINE 1 - TRAIN + EVALUATE the probe on the frozen V-JEPA 2.1 features (clean video)
#    Loads every feature shard from Drive, trains the attentive probe (3 seeds) and logs EVERY epoch.
# =====================================================================================================
import torch.nn as nn
r = rc(["copy", DRIVE_FEATS, FEATS_LOCAL, "--transfers", "8"], timeout=3600)
print("pulled feature shards from Drive | rc =", r.returncode, r.stderr[-150:], flush=True)

def _load(p):
    try: return torch.load(p, map_location="cpu", weights_only=True)
    except TypeError: return torch.load(p, map_location="cpu")

by = {}
for p in sorted(glob.glob(f"{FEATS_LOCAL}/*.pt")):
    s = _load(p); d = by.setdefault(s["split"], dict(x=[], vid=[], **{k: [] for k in s["meta"]}))
    d["x"].append(s["x"]); d["vid"] += [s["video_id"]] * len(s["x"])
    for k, v in s["meta"].items(): d[k] += v
DATA = {}
for sp, d in by.items():
    vc, nc = np.array(d["verb_class"]), np.array(d["noun_class"])
    DATA[sp] = dict(X=torch.cat(d["x"]), vid=np.array(d["vid"]), verb=np.array(d["verb"]), noun=np.array(d["noun"]), t=np.array(d["t"]),
                    Y=np.stack([vc, nc, np.array([ACTION_MAP.get((int(v), int(n)), -1) for v, n in zip(vc, nc)])], 1).astype(np.int64))
    d["x"].clear()
assert "train" in DATA and any(k.startswith("val") for k in DATA), "Need at least one train and one validation split - run cell 4 first."
VALSETS = {k: v for k, v in DATA.items() if k.startswith("val")}
PRIMARY = "val_heldout" if "val_heldout" in VALSETS else next(iter(VALSETS))
SUMMARY = pd.DataFrame([dict(split=k, videos=len(set(v["vid"])), clips=len(v["Y"]), feat_GB=round(v["X"].numel() * 2 / 1e9, 2),
                             unseen_action_pct=round(100 * float((v["Y"][:, 2] < 0).mean()), 1)) for k, v in DATA.items()])
print("\nDATA SUMMARY\n", SUMMARY.to_string(index=False), f"\nprimary validation set: {PRIMARY}", flush=True)
SUMMARY.to_csv(f"{RUN_DIR}/tables/data_summary.csv", index=False)

# ------------------------------ metrics (numpy) ------------------------------
HEADS = ("verb", "noun", "action")
def topk_hits(logits, labels, k):
    k = min(k, logits.shape[1]); idx = np.argpartition(-logits, k - 1, axis=1)[:, :k]
    return (idx == labels[:, None]).any(1) & (labels >= 0)               # unseen classes (-1) always count as a miss
def mean_class(hit, labels):
    cls = np.unique(labels[labels >= 0]); return float(np.mean([hit[labels == c].mean() for c in cls])) if len(cls) else 0.0
def metrics_np(logits3, Y, k=5):
    out = {}
    for h, lg, lb in zip(HEADS, logits3, Y.T):
        hit = topk_hits(lg, lb, k); out[f"{h}_r{k}"] = 100 * float(hit.mean()); out[f"{h}_mc{k}"] = 100 * mean_class(hit, lb)
    return out

# ------------------------------ models ------------------------------
class Block(nn.Module):
    def __init__(s, d, h):
        super().__init__(); s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.attn = nn.MultiheadAttention(d, h, batch_first=True)
        s.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(s, x):
        h = s.n1(x); x = x + s.attn(h, h, h, need_weights=False)[0]; return x + s.mlp(s.n2(x))

class AttProbe(nn.Module):                      # attentive probe: 2 self-attention blocks + 3 learned queries (verb / noun / action)
    def __init__(s, din, nv, nn_, na, d=768, h=12, depth=2):
        super().__init__()
        s.proj = nn.Sequential(nn.LayerNorm(din), nn.Linear(din, d)); s.blocks = nn.ModuleList([Block(d, h) for _ in range(depth)])
        s.q = nn.Parameter(torch.randn(3, d) * 0.02); s.cross = nn.MultiheadAttention(d, h, batch_first=True)
        s.norm = nn.LayerNorm(d); s.drop = nn.Dropout(0.1); s.heads = nn.ModuleList([nn.Linear(d, nv), nn.Linear(d, nn_), nn.Linear(d, na)])
    def forward(s, t):
        x = s.proj(t)
        for b in s.blocks: x = b(x)
        p = s.drop(s.norm(s.cross(s.q.unsqueeze(0).expand(x.size(0), -1, -1), x, x, need_weights=False)[0]))
        return [hd(p[:, i]) for i, hd in enumerate(s.heads)]

class LinProbe(nn.Module):                      # simple comparison: linear heads on the mean-pooled tokens
    def __init__(s, din, nv, nn_, na):
        super().__init__(); s.norm = nn.LayerNorm(din); s.heads = nn.ModuleList([nn.Linear(din, nv), nn.Linear(din, nn_), nn.Linear(din, na)])
    def forward(s, t): p = s.norm(t.mean(1)); return [h(p) for h in s.heads]

USE_AMP = DEVICE == "cuda"
def predict(model, X, sl=slice(None), bs=128):
    model.eval(); outs = [[], [], []]
    with torch.no_grad():
        for i in range(0, len(X), bs):
            x = X[i:i + bs][:, sl].to(DEVICE).float()
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP): lg = model(x)
            for o, l in zip(outs, lg): o.append(l.float().cpu().numpy())
    return [np.concatenate(o) for o in outs]

def train_probe(arch, seed, k_steps=None, epochs=EPOCHS, tag="attentive", verbose=True):
    """Train one probe on the TRAIN features; evaluate on every validation set after EVERY epoch.
       k_steps = use only the last k temporal steps (each step = 0.5 s of video) -> context-length ablation."""
    torch.manual_seed(seed); np.random.seed(seed)
    sl = slice((TP - k_steps) * TPS, None) if k_steps else slice(None)
    Xtr, Ytr = DATA["train"]["X"], torch.from_numpy(DATA["train"]["Y"])
    model = (AttProbe if arch == "attentive" else LinProbe)(Xtr.shape[-1], N_VERB, N_NOUN, N_ACTION).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD); spe = max(len(Xtr) // PROBE_BS, 1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=epochs * spe, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP); hist = []
    for ep in range(epochs):
        t0 = time.time(); model.train(); perm = torch.randperm(len(Xtr)); run = 0.0
        for st in range(spe):
            idx = perm[st * PROBE_BS:(st + 1) * PROBE_BS]
            x, y = Xtr[idx][:, sl].to(DEVICE).float(), Ytr[idx].to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP): lv, ln, la = model(x)
            loss = F.cross_entropy(lv.float(), y[:, 0]) + F.cross_entropy(ln.float(), y[:, 1])
            m = y[:, 2] >= 0
            if m.any(): loss = loss + F.cross_entropy(la.float()[m], y[m, 2])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        row = dict(tag=tag, seed=seed, k_steps=k_steps or TP, epoch=ep + 1, train_loss=run / spe)
        for vs, v in VALSETS.items():
            for k_, val in metrics_np(predict(model, v["X"], sl), v["Y"]).items(): row[f"{vs}/{k_}"] = val
        row["secs"] = time.time() - t0; hist.append(row)
        if verbose:
            print(f"[{tag} seed {seed}] epoch {ep + 1:02d}/{epochs} loss {row['train_loss']:.3f} | {PRIMARY} R@5: verb {row[f'{PRIMARY}/verb_r5']:.1f} "
                  f"noun {row[f'{PRIMARY}/noun_r5']:.1f} action {row[f'{PRIMARY}/action_r5']:.1f} | {row['secs']:.0f}s", flush=True)
    return model, hist

# ------------------------------ main attentive probe, one run per seed ------------------------------
HIST, MODELS = [], {}
for sd in SEEDS:
    model, h = train_probe("attentive", sd); HIST += h; MODELS[sd] = model
HIST_DF = pd.DataFrame(HIST); HIST_DF.to_csv(f"{RUN_DIR}/tables/epoch_history_attentive.csv", index=False)
LOGITS0 = {vs: predict(MODELS[SEEDS[0]], v["X"]) for vs, v in VALSETS.items()}               # final-epoch logits of the first seed
np.savez_compressed(f"{RUN_DIR}/val_logits_seed{SEEDS[0]}.npz", **{f"{vs}_{h}": l.astype(np.float16) for vs, ls in LOGITS0.items() for h, l in zip(HEADS, ls)})
torch.save(MODELS[SEEDS[0]].state_dict(), f"{RUN_DIR}/probe_attentive_seed{SEEDS[0]}.pt")

FINAL_ROWS = HIST_DF[HIST_DF.epoch == EPOCHS]
print("\nFINAL-EPOCH RESULTS, attentive probe on frozen V-JEPA 2.1 (clean video), mean ± std over seeds %s:" % SEEDS)
for vs in VALSETS:
    print(f"  {vs}:  " + " | ".join(f"{h} R@5 {FINAL_ROWS[f'{vs}/{h}_r5'].mean():.1f}±{FINAL_ROWS[f'{vs}/{h}_r5'].std(ddof=0):.1f} (mean-class {FINAL_ROWS[f'{vs}/{h}_mc5'].mean():.1f})" for h in HEADS))
sync_run("baseline-1 training results")

### 6) Evaluation tests: comparisons, context-length ablation, top-k, per-class, per-video, confidence intervals

In [ ]:
# =====================================================================================================
# 6) EVALUATION TESTS for baseline 1: comparisons, context-length ablation, top-k, per-class, per-video, confidence intervals
# =====================================================================================================
NCLS = dict(verb=N_VERB, noun=N_NOUN, action=N_ACTION)

def final_of(hist_df, tag, vs, seeds=None):
    """final-epoch metrics of a tagged run -> {metric: (mean, std)} over seeds"""
    d = hist_df[(hist_df.tag == tag) & (hist_df.epoch == EPOCHS)]
    if seeds is not None: d = d[d.seed.isin(seeds)]
    return {m: (d[f"{vs}/{m}"].mean(), d[f"{vs}/{m}"].std(ddof=0)) for m in [f"{h}_{s}" for h in HEADS for s in ("r5", "mc5")]}

# ---- reference points that need no model ----
def chance_metrics():
    return {f"{h}_{s}": (100 * min(5, NCLS[h]) / NCLS[h], 0.0) for h in HEADS for s in ("r5", "mc5")}
def freq_prior_metrics(vs):
    Ytr, Yv = DATA["train"]["Y"], DATA[vs]["Y"]; out = {}
    for j, h in enumerate(HEADS):
        cnt = np.bincount(Ytr[Ytr[:, j] >= 0, j], minlength=NCLS[h]); top5 = np.argsort(-cnt)[:5]
        hit = np.isin(Yv[:, j], top5) & (Yv[:, j] >= 0)
        out[f"{h}_r5"] = (100 * float(hit.mean()), 0.0); out[f"{h}_mc5"] = (100 * mean_class(hit, Yv[:, j]), 0.0)
    return out

# ---- linear probe on mean-pooled tokens (3 seeds) and context-length ablation (last 0.5 s / 1 s / 2 s of the 4 s window) ----
EXTRA = []
for sd in SEEDS: EXTRA += train_probe("linear", sd, tag="linear_meanpool", verbose=(sd == SEEDS[0]))[1]
K_STEPS = [k for k in (1, 2, 4) if k < TP]
for k in K_STEPS: EXTRA += train_probe("attentive", SEEDS[0], k_steps=k, tag=f"context_{k}", verbose=(k == K_STEPS[0]))[1]
ALL_HIST = pd.concat([HIST_DF, pd.DataFrame(EXTRA)], ignore_index=True)
ALL_HIST.to_csv(f"{RUN_DIR}/tables/epoch_history_all.csv", index=False)

# ---- comparison table (every validation set) ----
def fmt(v): return f"{v[0]:.1f}" if v[1] == 0 else f"{v[0]:.1f} ± {v[1]:.1f}"
METHODS = {"Chance (uniform guess)": lambda vs: chance_metrics(),
           "Frequency prior (top-5 train classes)": freq_prior_metrics,
           "Linear probe, mean-pooled tokens": lambda vs: final_of(ALL_HIST, "linear_meanpool", vs),
           "Attentive probe, full 4 s (baseline 1)": lambda vs: final_of(ALL_HIST, "attentive", vs)}
LONG, COMPARE = [], {}
for vs in VALSETS:
    rows = []
    for name, fn in METHODS.items():
        m = fn(vs); rows.append({"method": name, **{k: fmt(v) for k, v in m.items()}})
        LONG += [dict(valset=vs, method=name, metric=k, mean=v[0], std=v[1]) for k, v in m.items()]
    COMPARE[vs] = pd.DataFrame(rows)
LONG = pd.DataFrame(LONG); LONG.to_csv(f"{RUN_DIR}/tables/comparison_long.csv", index=False)
for vs, df in COMPARE.items():
    df.to_csv(f"{RUN_DIR}/tables/comparison_{vs}.csv", index=False)
    print(f"\nCOMPARISON on {vs} (Recall@5 %, mean ± std over seeds; mc5 = mean-class Recall@5)\n",
          df[["method", "verb_r5", "noun_r5", "action_r5", "verb_mc5", "noun_mc5", "action_mc5"]].to_string(index=False), flush=True)
print("\nReference (from the project proposal): V-JEPA 2.1 paper reports 40.8 Recall@5 for action anticipation with full training "
      "and the official protocol - not directly comparable to this reduced-data pilot.")

# ---- context-length ablation ----
CTX = []
for vs in VALSETS:
    for k in K_STEPS + [TP]:
        tag = "attentive" if k == TP else f"context_{k}"
        d = ALL_HIST[(ALL_HIST.tag == tag) & (ALL_HIST.epoch == EPOCHS) & (ALL_HIST.seed == SEEDS[0])]
        CTX.append(dict(valset=vs, seconds=k * OBS_SEC / TP, **{m: float(d[f"{vs}/{m}"].iloc[0]) for m in ("verb_r5", "noun_r5", "action_r5")}))
CTX = pd.DataFrame(CTX); CTX.to_csv(f"{RUN_DIR}/tables/ablation_context_length.csv", index=False)
print("\nCONTEXT-LENGTH ABLATION (seed %d, Recall@5 %%; only the last N seconds of the 4 s window are visible to the probe)\n" % SEEDS[0],
      CTX[CTX.valset == PRIMARY].round(1).to_string(index=False))

# ---- top-k curves, per-class, per-video (attentive probe, first seed) ----
TOPK = pd.DataFrame([dict(valset=vs, head=h, k=k, recall=100 * float(topk_hits(LOGITS0[vs][j], VALSETS[vs]["Y"][:, j], k).mean()))
                     for vs in VALSETS for j, h in enumerate(HEADS) for k in range(1, 11)])
TOPK.to_csv(f"{RUN_DIR}/tables/topk_curve.csv", index=False)

PERCLASS = []
for j, h in enumerate(("verb", "noun")):
    lb, names = VALSETS[PRIMARY]["Y"][:, j], VALSETS[PRIMARY][h]; hit = topk_hits(LOGITS0[PRIMARY][j], lb, 5)
    for c in pd.Series(lb).value_counts().head(15).index:
        PERCLASS.append(dict(head=h, cls=int(c), name=str(names[lb == c][0]), val_clips=int((lb == c).sum()), train_clips=int((DATA["train"]["Y"][:, j] == c).sum()),
                             recall5=100 * float(hit[lb == c].mean())))
PERCLASS = pd.DataFrame(PERCLASS); PERCLASS.to_csv(f"{RUN_DIR}/tables/per_class_top15.csv", index=False)

PERVIDEO = []
for vs, v in VALSETS.items():
    hits = [topk_hits(LOGITS0[vs][j], v["Y"][:, j], 5) for j in range(3)]
    for vid in sorted(set(v["vid"])):
        m = v["vid"] == vid; PERVIDEO.append(dict(valset=vs, vid=vid, clips=int(m.sum()), **{f"{h}_r5": 100 * float(hits[j][m].mean()) for j, h in enumerate(HEADS)}))
PERVIDEO = pd.DataFrame(PERVIDEO); PERVIDEO.to_csv(f"{RUN_DIR}/tables/per_video.csv", index=False)

# ---- bootstrap 95% confidence intervals over clips (attentive probe, first seed) ----
rng = np.random.default_rng(0); CI = []
for vs, v in VALSETS.items():
    for j, h in enumerate(HEADS):
        hit = topk_hits(LOGITS0[vs][j], v["Y"][:, j], 5).astype(float)
        bs = [hit[rng.integers(0, len(hit), len(hit))].mean() * 100 for _ in range(1000)]
        CI.append(dict(valset=vs, head=h, recall5=100 * hit.mean(), ci_low=np.percentile(bs, 2.5), ci_high=np.percentile(bs, 97.5), n_clips=len(hit)))
CI = pd.DataFrame(CI); CI.to_csv(f"{RUN_DIR}/tables/bootstrap_ci.csv", index=False)
print("\n95% bootstrap confidence intervals (Recall@5 %, over clips)\n", CI.round(1).to_string(index=False), flush=True)
sync_run("evaluation tables")

### 7) Figures

In [ ]:
# =====================================================================================================
# 7) FIGURES  -  everything is shown here and saved as PNG in <run>/figures (backed up to Drive)
# =====================================================================================================
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 170, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10, "axes.titlesize": 11})
HC = dict(verb="#4C78A8", noun="#F58518", action="#54A24B")
VS_STYLE = {"val_heldout": ("held-out videos (HF mirror)", "-"), "val_official": ("official val videos (Bristol)", "--")}
vs_name = lambda vs: VS_STYLE.get(vs, (vs, "-"))[0]
FOOT = f"{RUN_ID} | train clips {len(DATA['train']['Y'])}, " + ", ".join(f"{vs} {len(v['Y'])}" for vs, v in VALSETS.items()) + f" | seeds {SEEDS}"
FIGS = []
def finish(fig, name, foot=True):
    if foot: fig.text(0.5, -0.01, FOOT, ha="center", fontsize=7, color="gray")
    fig.tight_layout(); path = f"{RUN_DIR}/figures/{name}.png"; fig.savefig(path, bbox_inches="tight"); FIGS.append(path); plt.show(); plt.close(fig)

def band(ax, df, col, **kw):
    g = df.groupby("epoch")[col].agg(["mean", "std"]).fillna(0); ax.plot(g.index, g["mean"], **kw)
    ax.fill_between(g.index, g["mean"] - g["std"], g["mean"] + g["std"], color=kw.get("color"), alpha=0.18)

# ---- 1) per-epoch curves (attentive probe, mean ± std over seeds) ----
A = ALL_HIST[ALL_HIST.tag == "attentive"]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
band(ax[0], A, "train_loss", color="#555", label="train loss"); ax[0].set(title="Training loss (verb + noun + action CE)", xlabel="epoch", ylabel="loss")
for h in HEADS:
    for vs in VALSETS:
        band(ax[1], A, f"{vs}/{h}_r5", color=HC[h], ls=VS_STYLE.get(vs, ("", "-"))[1], label=f"{h} ({'held-out' if vs == 'val_heldout' else 'official'})")
        band(ax[2], A, f"{vs}/{h}_mc5", color=HC[h], ls=VS_STYLE.get(vs, ("", "-"))[1], label=f"{h} ({'held-out' if vs == 'val_heldout' else 'official'})")
ax[1].set(title="Validation Recall@5 per epoch", xlabel="epoch", ylabel="%"); ax[2].set(title="Validation mean-class Recall@5 per epoch", xlabel="epoch", ylabel="%")
ax[1].legend(fontsize=7, ncol=2); finish(fig, "fig01_epoch_curves")

# ---- 2) method comparison (bars with std over seeds) ----
short = {"Chance (uniform guess)": "chance", "Frequency prior (top-5 train classes)": "freq. prior",
         "Linear probe, mean-pooled tokens": "linear probe", "Attentive probe, full 4 s (baseline 1)": "attentive probe (baseline 1)"}
mcol = ["#bbbbbb", "#8c8c8c", "#6baed6", "#08519c"]
fig, axes = plt.subplots(2, len(VALSETS), figsize=(7.4 * len(VALSETS), 8), squeeze=False)
for c, vs in enumerate(VALSETS):
    for r_, (suf, lab) in enumerate((("r5", "Recall@5"), ("mc5", "mean-class Recall@5"))):
        ax, w = axes[r_][c], 0.2
        for mi, mname in enumerate(METHODS):
            d = LONG[(LONG.valset == vs) & (LONG.method == mname)].set_index("metric")
            vals = [d.loc[f"{h}_{suf}", "mean"] for h in HEADS]; errs = [d.loc[f"{h}_{suf}", "std"] for h in HEADS]
            b = ax.bar(np.arange(3) + (mi - 1.5) * w, vals, w, yerr=errs, capsize=2, color=mcol[mi], label=short[mname])
            for x_, v_ in zip(np.arange(3) + (mi - 1.5) * w, vals): ax.text(x_, v_ + 0.5, f"{v_:.1f}", ha="center", fontsize=6.5, rotation=90)
        ax.set_xticks(range(3)); ax.set_xticklabels([h.capitalize() for h in HEADS]); ax.set_ylabel("%")
        ax.set_title(f"{lab} - {vs_name(vs)}"); ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
        if r_ == 0 and c == 0: ax.legend(fontsize=8, loc="upper right")
finish(fig, "fig02_method_comparison")

# ---- 3) context-length ablation ----
fig, ax = plt.subplots(figsize=(6.6, 4.4))
for vs in VALSETS:
    for h in HEADS:
        d = CTX[CTX.valset == vs].sort_values("seconds"); ax.plot(d.seconds, d[f"{h}_r5"], marker="o", color=HC[h], ls=VS_STYLE.get(vs, ("", "-"))[1],
                                                                label=f"{h} ({'held-out' if vs == 'val_heldout' else 'official'})")
ax.set(xlabel="visible context before the anticipation point (s)", ylabel="Recall@5 (%)", title="Context-length ablation: only the last N s of the 4 s window"); ax.legend(fontsize=7, ncol=2)
finish(fig, "fig03_context_ablation")

# ---- 4) top-k curves ----
fig, ax = plt.subplots(figsize=(6.6, 4.4))
for vs in VALSETS:
    for h in HEADS:
        d = TOPK[(TOPK.valset == vs) & (TOPK["head"] == h)]; ax.plot(d.k, d.recall, marker="o", ms=3, color=HC[h], ls=VS_STYLE.get(vs, ("", "-"))[1], label=f"{h} ({'held-out' if vs == 'val_heldout' else 'official'})")
ax.set(xlabel="k", ylabel="Recall@k (%)", title="Top-k recall of the attentive probe"); ax.set_xticks(range(1, 11)); ax.legend(fontsize=7, ncol=2)
finish(fig, "fig04_topk_curve")

# ---- 5) per-class recall (most frequent validation classes) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for ax, h in zip(axes, ("verb", "noun")):
    d = PERCLASS[PERCLASS["head"] == h].sort_values("val_clips")
    ax.barh([f"{n} (n={c})" for n, c in zip(d.name, d.val_clips)], d.recall5, color=HC[h]); ax.set(xlabel="Recall@5 (%)", title=f"Most frequent {h}s in {vs_name(PRIMARY)}")
finish(fig, "fig05_per_class_recall")

# ---- 6) per-video recall ----
fig, ax = plt.subplots(figsize=(max(8, 0.55 * len(PERVIDEO)), 4.6)); d = PERVIDEO.sort_values(["valset", "vid"]).reset_index(drop=True); w = 0.27
for i, h in enumerate(HEADS): ax.bar(d.index + (i - 1) * w, d[f"{h}_r5"], w, color=HC[h], label=h)
ax.set_xticks(d.index); ax.set_xticklabels([f"{v}\n({'H' if s == 'val_heldout' else 'O'}, n={n})" for v, s, n in zip(d.vid, d.valset, d.clips)], fontsize=6.5, rotation=90)
ax.set(ylabel="Recall@5 (%)", title="Per-video results (H = held-out video, O = official validation video)"); ax.legend()
finish(fig, "fig06_per_video")

# ---- 7) dataset overview ----
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
cnt = pd.concat([pd.DataFrame(dict(vid=v["vid"], split=sp)) for sp, v in DATA.items()]).groupby(["split", "vid"]).size().reset_index(name="clips")
cols = {"train": "#4C78A8", "val_heldout": "#E45756", "val_official": "#72B7B2"}
cnt = cnt.sort_values(["split", "clips"], ascending=[True, False]).reset_index(drop=True)
ax[0].bar(cnt.index, cnt.clips, color=[cols.get(s, "gray") for s in cnt.split]); ax[0].set(title="Clips per video", xlabel="video (grouped by split)", ylabel="clips")
ax[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in cols.values()], labels=list(cols), fontsize=7)
for h, j in (("verb", 0), ("noun", 1)):
    f = np.sort(np.bincount(DATA["train"]["Y"][:, j]))[::-1]; f = f[f > 0]; ax[1].loglog(np.arange(1, len(f) + 1), f, color=HC[h], label=h)
ax[1].set(title="Class frequency in the training clips (long tail)", xlabel="class rank", ylabel="clips"); ax[1].legend()
sp_clips = pd.Series({sp: len(v["Y"]) for sp, v in DATA.items()}); ax[2].bar(sp_clips.index, sp_clips.values, color=[cols.get(s, "gray") for s in sp_clips.index])
for i, v_ in enumerate(sp_clips.values): ax[2].text(i, v_, str(v_), ha="center", va="bottom")
ax[2].set(title="Clips per split", ylabel="clips"); finish(fig, "fig07_dataset_overview")

# ---- 8) download speed (this session) ----
_dl = f"{RUN_DIR}/tables/download_log.csv"
if os.path.exists(_dl) and len(pd.read_csv(_dl)):
    L = pd.read_csv(_dl); fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    for src, g_ in L.groupby("source"): ax[0].scatter(g_.gb, g_.dl_MBps, label=src, s=28)
    ax[0].set(title="Download speed per video", xlabel="video size (GB)", ylabel="MB/s"); ax[0].legend()
    t = L.groupby("source").agg(dl=("dl_secs", "sum"), enc=("encode_secs", "sum")); t.plot.bar(ax=ax[1], rot=0, color=["#F58518", "#4C78A8"])
    ax[1].set(title="Total time spent per source (s)", ylabel="seconds"); finish(fig, "fig08_download_speed")

# ---- 9) bootstrap confidence intervals ----
fig, axes = plt.subplots(1, len(VALSETS), figsize=(6.4 * len(VALSETS), 4.2), squeeze=False)
for ax, vs in zip(axes[0], VALSETS):
    d = CI[CI.valset == vs].set_index("head").loc[list(HEADS)]
    ax.bar(range(3), d.recall5, yerr=[d.recall5 - d.ci_low, d.ci_high - d.recall5], capsize=5, color=[HC[h] for h in HEADS])
    for i, (v_, n_) in enumerate(zip(d.recall5, d.n_clips)): ax.text(i, v_ + 0.8, f"{v_:.1f}", ha="center")
    ax.set_xticks(range(3)); ax.set_xticklabels([h.capitalize() for h in HEADS]); ax.set(ylabel="Recall@5 (%)", title=f"95% bootstrap CI - {vs_name(vs)} (n={int(d.n_clips.iloc[0])})")
finish(fig, "fig09_confidence_intervals")
print("saved figures:", [os.path.basename(p) for p in FIGS])
sync_run("figures")

### 8) Report and final backup to Google Drive

In [ ]:
# =====================================================================================================
# 8) REPORT + FINAL BACKUP  -  report.md (tables + figure index), results_summary.json, full copy of the run to Google Drive
# =====================================================================================================
def md_table(df):
    cols = list(df.columns); out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows(): out.append("| " + " | ".join(f"{v:.1f}" if isinstance(v, float) else str(v) for v in r.values) + " |")
    return "\n".join(out)

hl = {}
for vs in VALSETS:
    f = final_of(ALL_HIST, "attentive", vs); hl[vs] = {k: dict(mean=round(v[0], 2), std=round(v[1], 2)) for k, v in f.items()}
ci_note = CI[CI.valset == PRIMARY].round(1)[["head", "recall5", "ci_low", "ci_high", "n_clips"]]
cols = ["method", "verb_r5", "noun_r5", "action_r5", "verb_mc5", "noun_mc5", "action_mc5"]

R = ["# Baseline 1 - frozen V-JEPA 2.1 on clean video (EgoRecover-JEPA)", f"Run: `{RUN_ID}`  |  platform: {ENV}", "",
     "## Protocol",
     f"- Encoder: `{MODEL}` (frozen). Input: {NF} frames sampled at {FPS_X:g} fps from a {OBS_SEC:g} s window that ends {TAU_A:g} s before the action starts (EPIC-KITCHENS-100 anticipation setup), short side resized to {IMG} px + centre crop.",
     f"- Features: tokens average-pooled spatially by {POOL} -> {N_TOK} tokens per clip ({TP} time steps x {TPS} spatial), cached once per video (fp16).",
     f"- Trained part: attentive probe (2 self-attention blocks + 3 learned queries) with verb ({N_VERB}), noun ({N_NOUN}) and action ({N_ACTION}) heads; {EPOCHS} epochs, AdamW lr {LR}, wd {WD}, batch {PROBE_BS}; seeds {SEEDS}.",
     "- Metrics: Recall@5 and mean-class Recall@5. Verb-noun pairs never seen in the official train csv count as misses.",
     "- Validation sets: **held-out videos** = whole videos from the HuggingFace extension mirror never used for training (video-level split); "
     "**official val videos** = videos of the official EPIC-100 validation split downloaded from the Bristol server (if enabled).", "",
     "## Data used", md_table(SUMMARY), ""]
for vs in VALSETS:
    R += [f"## Results on {vs_name(vs)} (Recall@5 %, mean ± std over seeds)", md_table(COMPARE[vs][cols]), ""]
R += [f"## 95% bootstrap confidence intervals - {vs_name(PRIMARY)} (attentive probe, seed {SEEDS[0]})", md_table(ci_note), "",
      f"## Context-length ablation - {vs_name(PRIMARY)} (seed {SEEDS[0]})", md_table(CTX[CTX.valset == PRIMARY].drop(columns="valset").round(1)), "",
      "## Notes and limitations",
      "- Reduced-data pilot: the number of videos is a small subset of EPIC-KITCHENS-100, so absolute numbers are NOT comparable with published results "
      "(the V-JEPA 2.1 paper reports 40.8 action Recall@5 with full training and the official protocol, as cited in the project proposal).",
      "- Held-out videos are from the same 18 participants as the training videos (like the official split) but only cover extension videos; official val videos cover other recordings.",
      "- Epoch curves show the final-epoch model; no checkpoint selection on validation data was done.",
      "- Frames are sampled on a 4 fps grid from one decoding pass per video; the same sampling must be used for all later baselines (corrupted video).",
      "- The HuggingFace V-JEPA 2.1 weights are a community port of Meta's checkpoints.", "",
      "## Files", "- `figures/` fig01-fig09 (epoch curves, method comparison, context ablation, top-k, per-class, per-video, dataset, download speed, confidence intervals)",
      "- `tables/` plan, data summary, epoch histories, comparison tables, ablation, top-k, per-class, per-video, bootstrap CI, download log",
      f"- `probe_attentive_seed{SEEDS[0]}.pt`, `val_logits_seed{SEEDS[0]}.npz`, `console_log.txt`, `config.json`",
      f"- Feature shards: `{DRIVE_FEATS}` | raw videos: `{DRIVE_VIDEOS}`"]
open(f"{RUN_DIR}/report.md", "w").write("\n".join(R))
json.dump(dict(run_id=RUN_ID, config=CONFIG, env=ENV, primary_valset=PRIMARY, final_attentive=hl, context_ablation=CTX.round(2).to_dict("records"),
               data=SUMMARY.to_dict("records")), open(f"{RUN_DIR}/results_summary.json", "w"), indent=2)
print("\n".join(R[:12])); print("..."); print(f"\nreport written: {RUN_DIR}/report.md")

# ---- final full backup + verification that it arrived ----
sync_run("FINAL")
listing = rc(["lsf", DRIVE_RUN, "-R", "--files-only"], timeout=300).stdout.split()
print(f"\nBACKED UP to Google Drive: EgoRecover-JEPA/runs/{RUN_ID}  ({len(listing)} files)")
print("  figures:", sum(p.startswith("figures/") for p in listing), "| tables:", sum(p.startswith("tables/") for p in listing),
      "| report.md:", "report.md" in listing, "| console_log.txt:", "console_log.txt" in listing)
print(f"  feature shards on Drive: {len(rc(['lsf', DRIVE_FEATS, '--files-only']).stdout.split())} | raw videos on Drive: "
      f"{len([p for p in rc(['lsf', DRIVE_VIDEOS, '-R', '--files-only']).stdout.split() if p.upper().endswith('.MP4')])}")

In [ ]:
# ================= FULL LOSS REPORT: every epoch of every run -> tables + images (saved to Drive) =================
# Source priority: in-memory ALL_HIST (same session) -> SOURCE_CSV (path you set) -> newest local run -> newest run on Drive
import os, sys, glob, json, time, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
SOURCE_CSV = None            # e.g. "/kaggle/working/runs/<run>/tables/epoch_history_all.csv"
LOG_Y      = True            # log scale on the loss axis (clearer when the first epochs are very high)
DRIVE_RUNS = "gdrive:EgoRecover-JEPA/runs"

def _run(args, timeout=600): return subprocess.run(["rclone"] + args, capture_output=True, text=True, timeout=timeout)
H, run_name = None, globals().get("RUN_ID", "run")
if "ALL_HIST" in globals():
    H = ALL_HIST.copy(); src = "memory (this session)"
elif SOURCE_CSV and os.path.exists(SOURCE_CSV):
    H = pd.read_csv(SOURCE_CSV); run_name = SOURCE_CSV.split("/")[-3]; src = SOURCE_CSV
else:
    loc = sorted(glob.glob("/kaggle/working/runs/*/tables/epoch_history_all.csv") + glob.glob("/content/runs/*/tables/epoch_history_all.csv"))
    if loc: H = pd.read_csv(loc[-1]); run_name = loc[-1].split("/")[-3]; src = loc[-1]
    else:
        dirs = sorted(d.strip("/") for d in _run(["lsf", DRIVE_RUNS, "--dirs-only"]).stdout.split())
        for d in reversed(dirs):
            os.makedirs("/tmp/_hist", exist_ok=True)
            if _run(["copyto", f"{DRIVE_RUNS}/{d}/tables/epoch_history_all.csv", "/tmp/_hist/h.csv"]).returncode == 0 and os.path.exists("/tmp/_hist/h.csv"):
                H = pd.read_csv("/tmp/_hist/h.csv"); run_name = d; src = f"Drive: {d}"; break
assert H is not None, "No epoch history found (run cells 5-6 first, or set SOURCE_CSV)."
print("using epoch history from:", src, "| rows:", len(H), flush=True)

OUT = (f"{RUN_DIR}/loss_report" if "RUN_DIR" in globals() else f"/kaggle/working/loss_report_{run_name}"); os.makedirs(OUT, exist_ok=True)
def label(r):
    if r.tag == "attentive": return f"attentive s{int(r.seed)}"
    if r.tag == "linear_meanpool": return f"linear s{int(r.seed)}"
    return f"context {int(r.k_steps) * 0.5:g}s s{int(r.seed)}"
H["run"] = H.apply(label, axis=1)
valsets = sorted({c.split("/")[0] for c in H.columns if "/" in c})
prim = "val_heldout" if "val_heldout" in valsets else valsets[0]

# ---- tables ----
runs_ = list(dict.fromkeys(H["run"]))
eps = np.sort(H["epoch"].unique())
M = np.full((len(eps), len(runs_)), np.nan)
for j, r_ in enumerate(runs_):
    d_ = H[H["run"] == r_]; M[np.searchsorted(eps, d_["epoch"].values), j] = d_["train_loss"].values
W = pd.DataFrame(M, index=pd.Index(eps, name="epoch"), columns=runs_)
order = [c for c in W.columns if c.startswith("attentive")] + [c for c in W.columns if c.startswith("linear")] + [c for c in W.columns if c.startswith("context")]
W = W[order]; W.round(4).to_csv(f"{OUT}/loss_by_epoch_all_runs.csv")
A = H[H.tag == "attentive"]
S = A.groupby("epoch").agg(train_loss_mean=("train_loss", "mean"), train_loss_std=("train_loss", "std"))
for vs in valsets:
    for h in ("verb", "noun", "action"): S[f"{vs}/{h}_r5"] = A.groupby("epoch")[f"{vs}/{h}_r5"].mean()
S.round(3).to_csv(f"{OUT}/epoch_summary_attentive_mean_over_seeds.csv")
H.drop(columns=["run"]).to_csv(f"{OUT}/epoch_metrics_all_runs_full.csv", index=False)
open(f"{OUT}/loss_table.md", "w").write("| epoch | " + " | ".join(W.columns) + " |\n|" + "---|" * (len(W.columns) + 1) + "\n" +
    "\n".join(f"| {e} | " + " | ".join(f"{v:.3f}" for v in row) + " |" for e, row in W.iterrows()))
print(W.round(3).iloc[[0, 1, 2, 4, 9, 14, 19, 24, -1]].to_string(), "\n(full table: loss_by_epoch_all_runs.csv)", flush=True)

# ---- image 1: training loss of every run ----
fig, ax = plt.subplots(figsize=(10, 5.4))
for c in W.columns:
    kw = dict(color="#08519c", lw=1.8) if c.startswith("attentive") else dict(color="#F58518", lw=1.8) if c.startswith("linear") else dict(color="#54A24B", lw=1.2, ls="--")
    ax.plot(W.index, W[c], label=c, alpha=0.9, **kw)
if LOG_Y: ax.set_yscale("log")
ax.set(xlabel="epoch", ylabel="training loss (verb + noun + action cross-entropy)", title=f"Training loss per epoch - all runs\n{run_name}")
ax.grid(alpha=.25); ax.legend(fontsize=7, ncol=3); fig.tight_layout(); fig.savefig(f"{OUT}/loss_all_runs.png", dpi=170); plt.show(); plt.close(fig)

# ---- image 2: loss vs held-out accuracy (overfitting diagnostic, attentive probe, mean over seeds) ----
fig, ax = plt.subplots(figsize=(9, 5)); ax2 = ax.twinx()
ax.plot(S.index, S.train_loss_mean, color="#444", lw=2, label="train loss"); ax.fill_between(S.index, S.train_loss_mean - S.train_loss_std.fillna(0), S.train_loss_mean + S.train_loss_std.fillna(0), color="#444", alpha=.15)
for h, c in (("verb", "#4C78A8"), ("noun", "#F58518"), ("action", "#54A24B")): ax2.plot(S.index, S[f"{prim}/{h}_r5"], color=c, lw=1.6, label=f"{h} Recall@5")
ax.set(xlabel="epoch", ylabel="training loss"); ax2.set_ylabel("held-out Recall@5 (%)"); ax.grid(alpha=.25)
ax.set_title("Training loss vs held-out accuracy (attentive probe, mean over seeds)\nFalling loss with flat accuracy = overfitting")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="center right")
fig.tight_layout(); fig.savefig(f"{OUT}/loss_vs_heldout_accuracy.png", dpi=170); plt.show(); plt.close(fig)

# ---- image 3: the full table as a picture (main runs) ----
cols = [c for c in W.columns if c.startswith("attentive") or c.startswith("linear")][:8]
T = W[cols].copy(); T.insert(0, "attentive mean", W[[c for c in W.columns if c.startswith("attentive")]].mean(axis=1))
fig, ax = plt.subplots(figsize=(1.35 * (len(T.columns) + 1), 0.3 * (len(T) + 2))); ax.axis("off")
tb = ax.table(cellText=[[str(e)] + [f"{v:.3f}" for v in r] for e, r in T.iterrows()], colLabels=["epoch"] + list(T.columns), loc="center", cellLoc="center")
tb.auto_set_font_size(False); tb.set_fontsize(7.5); tb.scale(1, 1.15)
for j in range(len(T.columns) + 1): tb[0, j].set_facecolor("#dbe5f1")
ax.set_title(f"Training loss by epoch - {run_name}", fontsize=10); fig.savefig(f"{OUT}/loss_table.png", dpi=170, bbox_inches="tight"); plt.show(); plt.close(fig)

# ---- save to Drive ----
if "rc" in globals() and "DRIVE_RUN" in globals(): r = rc(["copy", OUT, f"{DRIVE_RUN}/loss_report"])
else: r = _run(["copy", OUT, f"{DRIVE_RUNS}/{run_name}/loss_report"])
print(f"\nfiles in {OUT}:", sorted(os.listdir(OUT)), "\nuploaded to Drive:", f"EgoRecover-JEPA/runs/{run_name}/loss_report", "| rc =", r.returncode)